# Processing Sentinel-1 TOPS interferogram with ISCE3 & COMPASS
<br>  

**Author:** Zhenli Tang, Zhang Yunjun, August 3-7, 2026 [EarthScope InSAR Short Course (ISCE+)](https://www.earthscope.org/event/2026-technical-course-insar-processing-and-analysis-isce/).

**Learning Goals**:
* Understand unique features of Sentinel-1 SAR data (TOPS mode and Burst ID)
* Use ISCE3 & COMPASS to generate InSAR products for Sentinel-1 SLCs
* Understand the various processing steps needed to generate a geocoded interferogram
* Visualize intermediate and final results

**Estimated time**: 2 hours + exercises, needs to use the **m6a.xlarge** instance.

---

## Contents

1. [Initial setup](#id-1-initial-setup)
2. [Overview of Sentinel-1 TOPS burst and tutorial](#id-2-overview-of-sentinel-1-tops-burst-and-tutorial)
3. [Download SAR & auxliary data](#id-3-download-sar-and-auxliary-data)
4. [Coregistration: burst-wide Geocoded SLC](#id-4-coregistration-in-burst-geocoded-slc)
5. [Interferometric processing: interferogram formation, stitching, unwrapping, etc.](#id-5-interferometric-processing)
6. [Save Processed Outputs](#id-6-save-processed-outputs)


## 1. Initial setup

The cell below performs the intial setup of the notebook and **must be run every time the notebook (re)starts**. It defines the processing location and check the example dataset.

In [ ]:
# === Standard library ===
import gc
import os
from datetime import datetime
from pathlib import Path
from urllib.request import urlretrieve

# === Third-party ===
import numpy as np
import yaml
from matplotlib import pyplot as plt
from osgeo import gdal
from scipy.ndimage import uniform_filter
plt.rcParams.update({'font.size': 12})

import snaphu
from compass.utils.iono import download_ionex

# === Local imports ===
import utils as ut

# ---------------------------------------------------------------------------
# Configuration -- dateset info [need to be modified for every dataset]
# ---------------------------------------------------------------------------
ref_date = '2024-09-15'
sec_date = '2024-10-09'
wsen = (-155.50, 19.15, -154.95, 19.55)   # POLYGON((-155.5 19.55,-155.5 19.15,-154.95 19.15,-154.95 19.55,-155.5 19.55))

ref_ymd = ref_date.replace('-', '')
sec_ymd = sec_date.replace('-', '')
work_dir = Path(f'~/data/Hawaii_S1_A124_{ref_ymd}_{sec_ymd}').expanduser()
work_dir.mkdir(parents=True, exist_ok=True)
os.chdir(work_dir)
print('Go to directory:', work_dir)

# ---------------------------------------------------------------------------
# Configuration -- data structure
# ---------------------------------------------------------------------------
burst_db_path = work_dir.parent / 's1-burst-db' / 'opera-burst-bbox-only.sqlite3'
slc_dir = work_dir / 'SLC'
orbit_dir = work_dir / 'orbits'
dem_path = work_dir / 'DEM' / 'cop_dem.tif'
tec_dir = work_dir / 'TEC'                # global ionospheric maps files
cslc_dir = work_dir / 'CSLC'              # coregistered SLC files
ifgram_dir = work_dir / 'interferogram'   # inteferogram files

# Create all needed directories
for d in [slc_dir, orbit_dir, tec_dir, cslc_dir, ifgram_dir]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Configuration -- processing parameters
# ---------------------------------------------------------------------------
rglks = 4               # number of looks in range direction
azlks = 2               # number of looks in azimuth direction
filt_strength = 0.5     # Goldstein filter strength

# Clear memory from any previous run
gc.collect()

## 2. Overview of Sentinel-1 TOPS burst and tutorial


### 2.1 Background on TOPS Mode

The TOPS acquisition strategy is different than conventional stripmap mode.
The first satellite to use TOPS mode ([De Zan and Guarnieri, 2006](https://doi.org/10.1109/TGRS.2006.873853)) operationally was Sentinel-1.
TOPS stands for Terrain Observation with Progressive Scans. As the name
indicates, the radar sensor performs a scan of the surface by electronically
steering the antenna beam from a backward-pointing along-track direction to a
forward-pointing along-track direction for a fixed range swath (also called a
subswath).

![TOPS mode](docs/tops_mode.png)

After a successful scan at that range extent, the antenna beam is electronically
rolled back to its initial position, the range swath is electronically directed
outward to a new area to increase coverage, and the next scan is made from
backward to forward along track at this new range swath. After a third scan
at a third range swath, the entire process is repeated to create a continuous
image as the satellite flies along. The timing of the scans is such that there
is a small geographic overlap between successive scans at a given range to
ensure continuous coverage. Each scan at a given range swath is known as a
"burst". When inspecting one of the downloaded SLC products you will note that
data are provided in 3 individual subswaths (IW1, IW2, IW3) each with a set of
bursts. All together they form a Sentinel-1 frame, typically ~250 x 250 km in
size. The bursts are approximately 20 km in length and overlap by 2 km. Due to
the TOPS acquisition mode, the overlap region in successive bursts is seen from
two different directions (forward looking and backward looking).

![TOPS bursts](docs/tops.png)

### 2.2 Global Sentinel-1 Burst ID Map

Sentinel-1 performs systematic acquisition of bursts in both IW and EW modes. The bursts overlap almost perfectly between different passes and are always located at the same place. This area of interest can be identified using the Burst ID, which has been added to the official metadata since the SAR processor S1-IPF 3.4, and can be calculated for all Sentinel-1 SLC products using the formula below. The Burst ID map can be used to defined a common spatial grid for Sentinel-1 data globally.

![](docs/burst_id_map.png)

The Burst ID includes three components: relative orbit, burst index and the subswath:

![](docs/burst_id_convention.png)

The burst index is computed from orbital timing parameters
(ESA Sentinel-1 Level 1 Detailed Algorithm Definition, §9):

$$\text{burst\_index} = 1 + \left\lfloor \frac{\Delta t_b - T_{pre}}{T_{beam}} \right\rfloor$$

$$\Delta t_b = t_b - t_{anx} + (r - 1) T_{orb}$$

| Symbol | Value | Description |
|:-------|:-----:|:------------|
| $t_b$ | — | Mid-burst sensing time (IW2 midpoint) |
| $t_{anx}$ | — | Ascending node crossing time |
| $r$ | 1–175 | Relative orbit (track) number |
| $T_{orb}$ | $12 \times 86400 / 175 \approx 5924.57$ s | Nominal orbital period |
| $T_{pre}$ | 2.299849 s (IW) | Preamble length |
| $T_{beam}$ | 2.758273 s (IW) | Beam cycle time (one burst interval) |

The ESA website provides detailed background information on Sentinel-1 products
and technical information of the TOPS sensors:  
+ https://sentiwiki.copernicus.eu/web/sentinel-1  
+ https://sentiwiki.copernicus.eu/web/s1-documents


### 2.3 Dependencies


| Category | Package | Purpose |
|:---------|:--------|:--------|
| **Core Processing** | `isce3` | InSAR processing engine |
| | `compass` | COregistered Multi-temPorAl Sar Slc — CSLC geocoding & ionospheric TEC download |
| | `snaphu` | Phase unwrapping (SNAPHU) |
| **Data Download** | `burst2safe` | Burst download & SAFE conversion |
| | `eof` | Orbit file download |
| | `sardem` | DEM download |
| **Stitching** | `gdal` | Burst stitching |

## 3. Download SAR and auxliary data

### 3.0 Prepare OPERA Sentinel-1 burst ID database

This OPERA burst ID database covers all Sentinel-1 data globally. The downloading/preparation only needs to done once.

In [ ]:
# Download the pre-built OPERA burst ID database from GitHub
if not burst_db_path.exists():
    print('Downloading pre-built OPERA burst ID database from GitHub...')
    os.makedirs(os.path.dirname(burst_db_path), exist_ok=True)
    BURST_DB_URL = 'https://github.com/opera-adt/burst_db/releases/download/v0.10.0/opera-burst-bbox-only.sqlite3'
    urlretrieve(BURST_DB_URL, burst_db_path)
    print('OPERA burst ID database downloaded successfully!')
else:
    print(f'OPERA burst ID database already exist at {burst_db_path}.')

### 3.1 Search and download burst SLC

The ASF vertex page (https://search.asf.alaska.edu/) offers a GUI to visually search for available Sentinel-1 data over your area of interest. Once you have found your data, you can download it from the GUI. ASF provides a bulk-download python script. To download the Sentinel-1 data from ASF, you must have a `NASA Earthdata` account and have that configured in your `$HOME/.netrc` file. See `2026-isceplus/0.5_Data_Search_and_Access/Data_Access_Accounts.ipynb` for more on the data access accounts.

Here we use the `Dataset --> S1 Bursts` on the ASF page to search the available Sentinel-1 data in burst level, using the [Sep 2024 Kīlauea eruption in and near the Nāpau Crater on the middle East Rift Zone of Kīlauea](https://www.usgs.gov/volcanoes/kilauea/science/september-2024-napau-eruption) as an example. As shown in the screenshot below, there are 7 bursts in the ascending orbit overlapping with the given area of interest. Some basic information of these bursts includes:
+ relative orbit (path) = 124
+ subswaths: IW2, IW3

![Burst Coverage and Study Area](docs/satellite_bbox.png)

In [ ]:
ext_str = " ".join([str(x) for x in wsen])
!burst2stack --rel-orbit 124 --all-anns --pols VV --swaths IW2 IW3 --start-date {ref_date} --end-date {ref_date} --extent {ext_str} --output-dir {slc_dir}
!burst2stack --rel-orbit 124 --all-anns --pols VV --swaths IW2 IW3 --start-date {sec_date} --end-date {sec_date} --extent {ext_str} --output-dir {slc_dir}


### 3.2 Download orbits

In [ ]:
!eof --search-path {slc_dir} --save-dir {orbit_dir} --force-asf

### 3.3 Download DEM and water mask

In [ ]:
# use a bounding box much larger than the specified AOI above
# to cover all downloaded burst SLCs for the intermediate processing
buf = 2  # degree
dem_wsen = (np.floor(wsen[0] - buf), np.floor(wsen[1] - buf/2), np.ceil(wsen[2] + buf), np.ceil(wsen[3] + buf/2))
dem_path.parent.mkdir(parents=True, exist_ok=True)

# download Copernicus DEM
dem_wsen_str = " ".join([str(x) for x in dem_wsen])
!sardem --bbox {dem_wsen_str} --output-type float32 --output-format GTiff --data-source COP -o {dem_path}

# download NASADEM water-body mask
ut.download_nasadem_water_mask(dem_wsen, dem_path.parent)


### 3.4 Download TEC files

Download the global ionospheric maps (GIM) products from NASA Earthdata at https://www.earthdata.nasa.gov/data/space-geodesy-techniques/gnss/atmospheric-products.

In [ ]:
# ---- Download TEC (ionospheric) files before CSLC processing ----
for date in (ref_date, sec_date):
    date_str = datetime.strptime(date, '%Y-%m-%d').strftime('%Y%m%d')
    tec_file = download_ionex(date_str, str(tec_dir), sol_code='jpl')
    print(f'  {date}: {Path(tec_file).name}')
print('TEC download complete.')


## 4. Coregistration in burst geocoded SLC

### 4.1 Prepare the configuration file

Like ISCE2's `topsApp.py` (whose inputs are set through an XML file), the
COMPASS CSLC workflow is driven by a run-configuration **YAML** file. Below is
the *complete* initial configuration written for each burst/date — every group
is shown explicitly rather than relying on hidden defaults. The template
mirrors the COMPASS defaults
([`s1_cslc_geo.yaml`](https://github.com/opera-adt/COMPASS/blob/main/src/compass/defaults/s1_cslc_geo.yaml))
and conforms to the validation schema
([`s1_cslc_geo_schemas.yaml`](https://github.com/opera-adt/COMPASS/blob/main/src/compass/schemas/s1_cslc_geo.yaml)).

```yaml
runconfig:
  name: cslc_s1_workflow_default
  groups:
    pge_name_group:
      pge_name: CSLC_S1_PGE
    input_file_group:
      safe_file_path:      # [SAFE file for this date]
      orbit_file_path:     # [orbit (EOF) file covering this date]
      burst_id:            # [burst ID, e.g. t124_264305_iw2]
    dynamic_ancillary_file_group:
      dem_file:            # DEM GeoTIFF
      dem_description: DEM description was not provided.
      tec_file:            # IONEX TEC file (optional; omitted if absent)
    static_ancillary_file_group:
      burst_database_file: # burst-db SQLite3
    product_path_group:
      product_path: .
      scratch_path: ./scratch
      sas_output_file:     # output HDF5 filename
      product_version: '0.2'
      product_specification_version: '0.1'
    primary_executable:
      product_type: CSLC_S1
    processing:
      polarization: co-pol
      geocoding:
        flatten: true
        x_posting: 5
        y_posting: 10
      geo2rdr:
        lines_per_block: 1000
        threshold: 1.0e-08
        numiter: 25
      correction_luts:
        enabled: true
        range_spacing: 120
        azimuth_spacing: 0.028
        troposphere:
          delay_type: wet_dry
      rdr2geo:
        threshold: 1.0e-08
        numiter: 25
        lines_per_block: 1000
        extraiter: 10
        compute_latitude: false
        compute_longitude: false
        compute_height: false
        compute_layover_shadow_mask: true
        compute_local_incidence_angle: true
        compute_ground_to_sat_east: true
        compute_ground_to_sat_north: true
    worker:
      internet_access: false
      gpu_enabled: false
      gpu_id: 0
    quality_assurance:
      browse_image:
        enabled: true
        complex_to_real: amplitude
        percent_low: 0
        percent_high: 95
        gamma: 0.5
        equalize: false
      perform_qa: true
      output_to_json: false
    output:
      cslc_data_type: complex64_zero_mantissa
      compression_enabled: true
      compression_level: 4
      chunk_size: [128, 128]
      shuffle: true
```

The `ut.write_geo_runconfig()` helper writes this YAML for each burst/date;
input paths are resolved automatically by `ut.find_burst_input_files()`.


Scan the downloaded SAFE directories to automatically populate
`burst_id_list` and `date_list` from annotation metadata.

In [ ]:
# find all burst ID and dates suitable for InSAR
burst_id_list = ut.find_burst_ids(slc_dir, orbit_dir, verbose=True)[0]

# write config file in YAML format: one file per burst/date
# reduce the output resolution from 5/10 m (default) to 10/20 m to speedup
kwargs = dict(product_path=cslc_dir, x_posting=10, y_posting=20)
config_list = []
for date_str in (ref_date, sec_date):
    date_str = date_str.replace('-', '')
    for burst_id in burst_id_list:
        # grab all input files
        safe_path, orbit_path, tec_path = ut.find_burst_input_files(
            date_str, burst_id, slc_dir, orbit_dir, tec_dir,
        )
        # write run-config file
        config_path = cslc_dir / 'runconfigs' / f'geo_runconfig_{date_str}_{burst_id}.yaml'
        ut.write_geo_runconfig(
            config_path, safe_path, orbit_path, burst_id, dem_path, burst_db_path, tec_path, **kwargs,
        )
        config_list.append(config_path)
        print(f'write: {config_path.name}')


### 4.2 Run CSLC coregistration

Run `COMPASS` to coregister all Sentinel-1 bursts into a pre-defined UTM grid (default: 5 m x 10 m spacing).

Output is OPERA-format HDF5 containing `VV`, `x_coordinates`,
`y_coordinates`, and `projection`.

In [ ]:
# parallel processing
# set up to 4 workers due to the CPU/memory limits of the m6a.xlarge instance.
parallel = True

if not parallel:
    # sequential processing
    for config_path in config_list:
        !s1_cslc.py --grid geo {config_path}

else:
    # parallel processing
    ut.run_s1_cslc_parallel(config_list, n_workers=4)

#### 4.2.1 Principle geocoded SLC processing

For an imaging radar, the location of an arbitrary pixel is
determined by the intersection of the centroid of the radar
beam with the ground surface. In a geocentric-Cartesian
coordinate system, this intersection can be described
by the range-Doppler equation as:

![](docs/geolocation.png)

The CSLC-S1 workflow uses range-Doppler terrain correction to geocode
each burst from radar coordinates (slant range × azimuth time) onto a
common UTM grid. For each output pixel, the algorithm:

1. Converts the UTM (x, y) coordinate to ECEF using the DEM height
2. Solves the range-Doppler equations to find the radar position
3. Applies timing correction LUTs to adjust the position
4. Interpolates the radar SLC at the corrected position
5. Removes azimuth carrier phase and topographic flattening phase

This workflow coregister each burst SLC into a pre-defined geographic grid ([Zheng and Zebker, 2017](https://doi.org/10.1109/JSTARS.2017.2697861); [Zebker, 2017](https://doi.org/10.1109/LGRS.2017.2753580))
using geometry with model-driven refinements ([Yunjun et al., 2022](https://doi.org/10.1109/TGRS.2022.3168509)).
This is different from the traditional workflow (as used in ISCE-2), which first
coregister the secondary SLC into the reference SLC using geometry with data-driven 
refinements such as cross-correlation or enhanced spectral diversity ([Fattahi et al., 2017](https://doi.org/10.1109/TGRS.2016.2614925)), then geocode 
the derived products into the geographic coordinate. Detailed algorithms can be found
in [OPERA S1-CSLC Algotirhm Theoretic Basis Document](https://cumulus.asf.earthdatacloud.nasa.gov/PUBLIC/DATA/OPERA/OPERA_CSLC-S1_ATBD_D-108752_Initial_2024-06-24_signed.pdf).

![](docs/magic.png)


**HDF5 Dataset Layout**

The OPERA CSLC-S1 HDF5 file follows a hierarchical structure consisting of the root group (`/`) and four top-level groups:

| Group | Description |
|:------|:------------|
| `/` | Root group containing main data raster layers: `VV` (complex64), `azimuth_carrier_phase`, `flattening_phase`, and geographical information: `x_coordinates`, `y_coordinates`, `projection` |
| `/identification` | File-level metadata including burst ID, mission identifier, orbit/timing parameters, product version, and bounding polygon |
| `/metadata` | Processing metadata: `orbit` (state vectors), `calibration_information`, `noise_information`, `processing_information` (timing corrections, runconfig, parameters) |
| `/quality_assurance` | Per-dataset statistics (min, max, mean, standard deviation) for `VV` phase/power and each timing correction field |

**Key Datasets**

| Dataset Path | Type | Dimensions | Description |
|:-------------|:-----|:-----------|:------------|
| `/data/VV` | complex64 | `n_y` × `n_x` | Geocoded single-look complex SAR image |
| `/data/azimuth_carrier_phase` | float64 | `n_y` × `n_x` | Azimuth carrier phase (rad) |
| `/data/flattening_phase` | float64 | `n_y` × `n_x` | Flattening phase (rad) |
| `/data/x_coordinates` | float64 | `n_x` | UTM easting (m) |
| `/data/y_coordinates` | float64 | `n_y` | UTM northing (m) |
| `/data/projection` | int32 | scalar | EPSG code of the UTM projection |
| `/metadata/orbit/position_x` | float64 | `n_sv` | Orbit state vector X position (m) |
| `/metadata/orbit/velocity_x` | float64 | `n_sv` | Orbit state vector X velocity (m/s) |
| `/metadata/.../timing_corrections/slant_range` | float64 | `n_az` × `n_rg` | Slant-range LUT grid (m) |
| `/metadata/.../timing_corrections/zero_doppler_time` | float64 | `n_az` × `n_rg` | Azimuth zero-Doppler time LUT grid (s) |
| `/metadata/.../timing_corrections/bistatic_delay` | float64 | `n_az` × `n_rg` | Bistatic delay correction (s) |
| `/metadata/.../timing_corrections/los_solid_earth_tides` | float64 | `n_az` × `n_rg` | Solid Earth tide correction in LOS (m) |
| `/metadata/.../timing_corrections/los_ionospheric_delay` | float64 | `n_az` × `n_rg` | Ionospheric delay correction in LOS (m) |
| `/metadata/.../timing_corrections/wet_los_troposphere_delay` | float64 | `n_az` × `n_rg` | Wet troposphere delay correction in LOS (m) |
| `/metadata/.../timing_corrections/dry_los_troposphere_delay` | float64 | `n_az` × `n_rg` | Dry troposphere delay correction in LOS (m) |

#### 4.2.2 Plot results

Compare the raw SAFE burst in radar coordinates with the geocoded
CSLC output on the UTM grid.  We display the radar amplitude, geocoded
amplitude, and geocoded phase side by side for a single burst.


In [ ]:
# Inspect a single burst: raw SAFE (rdr) vs. geocoded H5 (geo)
# Uses ut.extract_burst_slc() to extract only the matching burst
# from the multi-burst SAFE measurement TIFF.

demo_burst_id = burst_id_list[0]   #'t124_264306_iw2'
demo_date = ref_date.replace('-','')
demo_cslc_path = cslc_dir / demo_burst_id / demo_date / f'{demo_burst_id}_{demo_date}.h5'
demo_safe_path = ut.find_burst_input_files(demo_date, demo_burst_id, slc_dir, orbit_dir, tec_dir)[0]

ut.plot_coregistration(demo_safe_path, demo_cslc_path, demo_burst_id, demo_date)

### 4.3 CSLC auxiliary information

CSLC geocoding involves two categories of operations:

- **Geo-registration** &mdash; six timing corrections improve the accuracy of
  the inverse map &rarr; radar coordinate mapping, so that each pixel is
  precisely located on the correct radar grid position;
- **Phase compensation** &mdash; deramping, reramping, and flattening
  operate on the complex SLC phase to remove the TOPS carrier and
  geometric phase while preserving deformation, atmosphere, and other
  geophysical signals.


The following table lists all datasets that represent pixel offsets
(timing corrections and LUT profiles).  Phase-only datasets
are excluded here because they do not shift pixel positions.

- **Physical** — geophysical effects (tides, ionosphere, troposphere, etc)
- **Focusing** — focusing artefacts

| Category | Dataset | Unit | Pixel factor |
|:---------|:--------|:----:|:-------------|
| **physical** | ``los_ionospheric_delay`` | m | / 120 |
| | ``los_static_tropospheric_delay`` | m | / 120 |
| | ``los_solid_earth_tides`` | m | / 120 |
| | ``azimuth_solid_earth_tides`` | s | / 0.028 |
| **focusing** | ``geometry_steering_doppler`` | m | / 120 |
| | ``bistatic_delay`` | s | / 0.028 |
| | ``azimuth_fm_rate_mismatch`` | s | / 0.028 |


#### 4.3.1 Physical Corrections

Geophysical effects that shift radar pixel positions:

- **``los_ionospheric_delay``** — Ionospheric path delay along LOS (m)
- **``los_static_tropospheric_delay``** — Reconstructed from COMPASS formula (not stored in HDF5):
  $ \frac{ZPD}{cos(inc)} * exp(\frac{-h}{H}) $
- **``los_solid_earth_tides``** — Solid Earth tide LOS displacement (m)
- **``azimuth_solid_earth_tides``** — Solid Earth tide azimuth displacement (s)

All datasets are displayed on the radar LUT grid
(x: slant range, y: zero-Doppler time) in pixel units.


In [ ]:
# ---- 3.3.1 Physical: geophysical corrections (pixel units) ----
aux_ds = ut.read_aux_dataset(demo_cslc_path, dem_path)
phy_ds_names = ['los_ionospheric_delay', 'los_static_tropospheric_delay', 'los_solid_earth_tides', 'azimuth_solid_earth_tides']
for ds_name in [x for x in phy_ds_names if x in aux_ds.keys()]:
    ut.plot_timing(demo_cslc_path, ds_name=ds_name, aux=aux_ds, burst_id=demo_burst_id, date_str=ref_date)


#### 4.3.2 Focusing Artefact Corrections

The approximation ESA used during the SAR focusing can introduce 
the following artefacts ([Gisinger et al., 2022](https://doi.org/10.1109/TGRS.2022.3194216)) below:

- **``geometry_steering_doppler``** — Doppler-induced range shift (m)
- **``bistatic_delay``** — Bistatic-to-monostatic azimuth offset (s)
- **``azimuth_fm_rate_mismatch``** — TOPS azimuth FM rate mismatch (s)

All datasets are displayed on the radar LUT grid in pixel units.
Range-direction values (m) are divided by 120 m; azimuth-direction
values (s) are divided by 0.028 s.


In [ ]:
img_ds_names = ['geometry_steering_doppler', 'bistatic_delay', 'azimuth_fm_rate_mismatch']
for ds_name in [x for x in img_ds_names if x in aux_ds.keys()]:
    ut.plot_timing(demo_cslc_path, ds_name=ds_name, aux=aux_ds, burst_id=demo_burst_id, date_str=ref_date)


## 5. Interferometric processing

We form interferograms **per burst** before stitching.  This avoids
mixing data from different bursts (which have different TOPS azimuth
steering directions) during the sliding-window coherence computation.
The updated pipeline is:

1. **Form burst interferogram** — cross-multiply ref×sec within each burst
2. **Stitch interferograms** — combine into a single continuous image
3. **Complex coherence** — sliding-window correlation on stitched SLCs
4. **Multilook & filter** — reduce speckle, adaptive phase filter
5. **Phase-sigma coherence** — phase-stability measure for unwrapping
6. **SNAPHU unwrapping** — recover absolute phase


### 5.1 Form burst interferograms

Before processing all bursts, let's compute a single-burst
interferogram and complex coherence as a demonstration.
We use ``demo_burst_id`` (the middle burst) with the
reference/secondary date pair.

$$\Large I = S_1 S_2^*$$

$$\Large \gamma=
\frac{E\{S_1S_2^*\}}
{\sqrt{E\{|S_1|^2\}E\{|S_2|^2\}}}
$$


In [ ]:
# inputs
demo_burst_id = burst_id_list[0]   #'t124_264306_iw2'
ref_h5 = cslc_dir / demo_burst_id / ref_ymd / f'{demo_burst_id}_{ref_ymd}.h5'
sec_h5 = cslc_dir / demo_burst_id / sec_ymd / f'{demo_burst_id}_{sec_ymd}.h5'

# read data
ref_arr, gt = ut.read_cslc_array(ref_h5)[:2]
sec_arr = ut.read_cslc_array(sec_h5)[0]
#ref_arr, sec_arr, gt = ut.align_cslc_pair(ref_arr, gt, sec_arr, sec_gt)
# convert NaN into a valid complex number [for complex coherence estimation]
flag_nodata = ~np.isfinite(ref_arr) | ~np.isfinite(sec_arr)
ref_arr[flag_nodata] = 0
sec_arr[flag_nodata] = 0

# form interferogram
burst_ifg = ref_arr * np.conj(sec_arr)
burst_pha = np.where(np.abs(burst_ifg) == 0, np.nan, np.angle(burst_ifg))

# calculate complex coherence (5x5 window)
ref_pow = (np.abs(ref_arr) ** 2).astype(np.float32)
sec_pow = (np.abs(sec_arr) ** 2).astype(np.float32)
ifg_sum = uniform_filter(burst_ifg, size=5, mode='constant') * 25
ref_sum = uniform_filter(ref_pow, size=5, mode='constant') * 25
sec_sum = uniform_filter(sec_pow, size=5, mode='constant') * 25
with np.errstate(invalid='ignore'):
    burst_coh = np.abs(ifg_sum) / np.sqrt(ref_sum * sec_sum)
burst_coh = np.nan_to_num(burst_coh, nan=0.0).clip(0.0, 1.0).astype(np.float32)

# delete intermediate variables
del ref_arr, sec_arr, ref_pow, sec_pow, ifg_sum, ref_sum, sec_sum


In [ ]:
# display
ext = ut.extent_utm(gt, burst_ifg.shape)
fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True, sharex=True)
ut.plot_phase(axes[0], burst_pha, extent=ext, title=f'{demo_burst_id}  interferometric phase)')
ut.plot_coherence(axes[1], burst_coh, extent=ext, title=f'{demo_burst_id}  complex coherence')
for ax in axes: ut.set_ax_utm(ax)


In [ ]:
burst_id_list = ut.find_burst_ids(slc_dir, orbit_dir, verbose=True)[0]

# loop for all burst-pairs
burst_ifg_list = []       # per-burst interferogram GeoTIFF paths
burst_coh_list = []       # per-burst coherence GeoTIFF paths

for i, burst_id in enumerate(burst_id_list):
    print('-'*50)
    print(f'forming {i+1}/{len(burst_id_list)} burst interferogram for {burst_id}...')
    # Build paths to OPERA CSLC HDF5 files
    #   cslc_dir / <burst_id> / YYYYMMDD / <burst_id>_YYYYMMDD.h5
    ref_h5 = cslc_dir / burst_id / ref_ymd / f'{burst_id}_{ref_ymd}.h5'
    sec_h5 = cslc_dir / burst_id / sec_ymd / f'{burst_id}_{sec_ymd}.h5'

    # Form interferogram + coherence; save as GeoTIFF; returns output paths
    # coh_win=5  →  5×5 sliding boxcar window (full resolution coherence)
    ifg_path, coh_path = ut.ifgram_and_coherence(ref_h5, sec_h5, burst_id, ifgram_dir, coh_win=5)

    burst_ifg_list.append(ifg_path)
    burst_coh_list.append(coh_path)


### 5.2 Stitch interferograms

Process each burst: read CSLC arrays, align, form the interferogram
and complex coherence, then blit into the pre-allocated stitched arrays.
Finally display the stitched interferogram and coherence.


In [ ]:
ut.clear_large_arrays()

# stitching
# bbox_wsen=None -> stitch the ENTIRE union extent of all bursts;
# bbox_wsen=wsen -> clip the output to the AOI wsen.
ifg, ifg_gt, proj_wkt, epsg_utm = ut.stitch_bursts(burst_ifg_list, bbox_wsen=wsen)
coh = ut.stitch_bursts(burst_coh_list, bbox_wsen=wsen)[0]

print(f'complete burst interferograms stitching for {len(burst_ifg_list)} burst(s).')


In [ ]:
# --- Display (pixel coordinates to show image dimensions) ---
ut.plot_pair(
    np.angle(ifg), coh, figsize=(10, 4),
    plot1=ut.plot_phase, kw1={'extent': ut.extent_pixel(ifg.shape), 'title': 'interferometric phase'},
    plot2=ut.plot_coherence, kw2={'extent': ut.extent_pixel(coh.shape), 'title': 'complex coherence'})

### 5.3 Multilooking & Filtering

In [ ]:
# Multilook the stitched interferogram
print(f'multilooking interferogram in {rglks} / {azlks} looks in range / azimuth directions...')
ifg_ml, gt_ml = ut.multilook_ifg(ifg, azlks, rglks, ifg_gt)

# Goldstein filter
print(f'apply Goldstein filter with the filtering strength of {filt_strength}...')
ifg_filt = ut.goldstein_filter(ifg_ml, alpha=filt_strength, no_data_value=0)


In [ ]:
# plot --- 3-panel display: full-res -> multilooked -> filtered ---
ut.plot_phase_triple(
    np.angle(ifg), np.angle(ifg_ml), np.angle(ifg_filt), 
    figsize=(12, 3),
    title1=f'Full-resolution',
    title2=f'+ Multilooking ({azlks}x{rglks})',
    title3=f'+ Goldstein filtering (alpha={filt_strength})',
    ext1=ut.extent_pixel(ifg.shape))

# Free full-res arrays no longer needed
#del ifg, ifg_ml

### 5.4 Estimate phase-sigma coherence

The interferometric phase is wrapped modulo $2\pi$.  Before
unwrapping with SNAPHU we estimate a **phase-sigma (phsig)**
coherence map that serves as a quality indicator.

**Phase-sigma algorithm**

1. Estimate local range and azimuth phase gradients via neighbour
   differencing and Gaussian convolution
2. Extract a local window around each pixel, subtract the gradient
   to deramp, compute weighted phase variance
3. Convert variance to correlation:

$$\Large \gamma_\sigma = \frac{1}{\sqrt{2 n_{lk} \sigma_\phi^2 + 1}}$$


In [ ]:
print('estimating phase-sigma coherence...')
phsig = ut.estimate_phsig_correlation(ifg_filt, ps_win=5, grad_win=5, nlks=rglks*azlks)

# --- Comparison: complex coherence vs phsig coherence ---
ut.plot_pair(
    coh, phsig,
    plot1=ut.plot_coherence, kw1={'extent': ut.extent_pixel(coh.shape), 'title': 'complex coherence'},
    plot2=ut.plot_coherence, kw2={'title': 'phase-sigma coherence', 'cbar_label': 'phsig coherence'},
)


### 5.5 Phase unwrapping

Unwrap the interferogram using the min-cost-flow method via [snaphu](https://github.com/isce-framework/snaphu-py).

In [ ]:
ifg_msk = ifg_filt.copy()

# [optional] mask out pixel on the water
# 1. load water mask into the same grid as the multilooked interferogram
water_mask_path = work_dir / 'DEM' / 'swbd_nasadem.wbd'
water_mask = ut.load_water_mask(gt_ml, ifg_filt.shape, epsg_utm, water_mask_path)
ifg_msk[water_mask] = 0

# unwrap
ncorrlooks = rglks * azlks / (1.2**2)
print(f'phase unwrapping with SNAPHU (ncorrlooks={ncorrlooks:.1f}, cost=smooth, init=mcf) ...')
unw, conncomp = snaphu.unwrap(ifg_msk, corr=phsig, nlooks=ncorrlooks, cost='smooth', init='mcf', nproc=1)

# Zero out invalid regions in output
unw[water_mask] = 0.0
conncomp[water_mask] = 0


In [ ]:
# --- Display wrapped & unwrapped phase over DEM hillshade ---
ut.plot_unwrap_results(ifg_filt, unw, conncomp, gt_ml, epsg_utm, dem_path, figsize=(12, 3))


## 6. Save processed outputs


### 6.1 Prepare LOS geometry information

Step to generate S1 Static layer Products if needed.

> ⏱ The following step generates **static layers** (layover/shadow mask, local incidence angle, LOS vectors) for each burst via COMPASS. This is **date-independent** (run once per burst) but **time-consuming** — each burst may take several minutes.
>
> **Recommended**: use ``ut.download_static_layers()`` in the next cell to download pre-computed CSLC-STATIC granules from ASF — this is much faster (typically < 1 min per burst vs. several minutes).  Falls back to local generation automatically.


In [ ]:
# ---- Static layers (one per burst, date-independent) ----
# Downloads pre-computed granules from ASF; falls back to
# local COMPASS generation for any burst not on ASF.

# Run s1_cslc.py --grid geo for every burst×date combination,
# then generate date-independent static layers (layover/shadow mask,
# incidence angle, LOS vectors) for each unique burst.

# 1. download existing OPERA static layers from ASF
exist_ids, missing_ids = ut.download_opera_static_layers(burst_id_list, work_dir, ref_ymd, bbox_wsen=wsen)

# 2. generate missing static layers using COMPASS
parallel = False
if len(missing_ids) > 0:
    print(f'\nGenerating remaining {len(missing_ids)} bursts locally...')

    # Write static-layers runconfigs (copy CSLC config, change product_type)
    geom_cfg_list = []
    for burst_id in missing_ids:

        # read an existing config file as reference
        src_cfg = cslc_dir / 'runconfigs' / f'geo_runconfig_{ref_ymd}_{burst_id}.yaml'
        with open(src_cfg) as f:
            cfg = yaml.safe_load(f)

        # write a new config file for static layer generation
        cfg['runconfig']['groups']['primary_executable']['product_type'] = 'CSLC_S1_STATIC'
        dst_cfg = cslc_dir / 'runconfigs' / f'static_layers_{burst_id}.yaml'
        with open(dst_cfg, 'w') as f:
            yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

        geom_cfg_list.append(dst_cfg)

    # run COMPASS to generate static layers
    # same parallel/sequential logic as CSLC
    if not parallel:
        for cfg_path in geom_cfg_list:
            !s1_cslc.py --grid geo {cfg_path}
    else:
        ut.run_s1_cslc_parallel(geom_cfg_list, n_workers=2)
else:
    print('All static layers exists.')


Save multilooked/filtered interferogram (phase), phase-sigma coherence,
unwrapped phase, and connected components to GeoTIFF files.


In [ ]:
# --- Compute LOS angles from static layers ---
# get static layer path list
static_layer_path_list = [str(cslc_dir / burst_id / ref_ymd / f'static_layers_{burst_id}.h5') for burst_id in burst_id_list]
# read geom transform from static layer
input_gt = ut.read_cslc_array(static_layer_path_list[0])[1]

print('Computing LOS angles from static layers ...')
inc_arr, az_arr, los_gt, los_epsg = ut.stitch_burst_los_angles(static_layer_path_list, input_gt=input_gt, out_gt=gt_ml, out_shape=ifg_filt.shape)
print(f'  LOS: {inc_arr.shape[0]} x {inc_arr.shape[1]}  epsg={los_epsg}')

In [ ]:
# --- Display LOS geometry in UTM coordinates ---
extent = ut.extent_utm(gt_ml, inc_arr.shape)
ut.plot_pair(
    inc_arr, az_arr, figsize=(10, 4),
    plot1=ut.plot_los, kw1={'extent': extent, 'title': 'LOS incidence angle'},
    plot2=ut.plot_los, kw2={'extent': extent, 'title': 'LOS azimuth angle'},
    coord='utm', epsg=los_epsg,
)


### 6.2 Write data to files

Write all relevant interferogram data into data files in GeoTIFF format.

In [ ]:
products = {
    'filt_mli.int.tif':            (ifg_filt,   gdal.GDT_CFloat32),
    'filt_mli.phsig.coh.tif':      (phsig,      gdal.GDT_Float32),
    'filt_mli.unw.tif':            (unw,        gdal.GDT_Float32),
    'filt_mli.unw.conncomp.tif':   (conncomp,   gdal.GDT_Int32),
    'los_incidence_angle.tif':     (inc_arr,    gdal.GDT_Float32),
    'los_azimuth_angle.tif':       (az_arr,     gdal.GDT_Float32),
}

for fname, (data, dtype) in products.items():
    path = ifgram_dir / fname
    ut.save_tiff(path, data, gt_ml, proj_wkt, dtype=dtype)
    print(f'Write file: {path}.')

## Notes on the ionosphere

In section 5.2, noticeable phase jumps can be observed at burst boundaries
in the stitched interferogram.  These phase discontinuities are caused by
**ionospheric effects** — spatial and temporal variations in the total
electron content (TEC) of the ionosphere introduce differential path delays
between the two acquisitions.

When ionospheric activity is low, these phase jumps become much less
pronounced. The figure below shows an interferogram for **2024-06-11 / 2024-06-23** 
where the ionosphere was relatively quiet. Notice that the inter-burst phase 
jumps are nearly absent.

![](docs/ifg_wo_phase_jump.png)

To correct for the ionospheric delay in the burst GSLC using the state-of-the-art
range split spectrum approach, one could generate the sub-band GSLC using COMPASS,
then follow the standard workflow as shown in [Liang et al. (2019)](https://doi.org/10.1109/TGRS.2019.2908494) to 
compute and remove the ionospheric delay.

# Homework

## The 2020 Mw 6.7 Elazığ Earthquake, Turkey

On 24 January 2020 at 20:55 UTC, an Mw 6.7 earthquake struck near the town of
Sivrice in Elazığ Province, eastern Turkey. The left-lateral strike-slip rupture
occurred along the Sivrice–Pütürge Segment of the East Anatolian Fault Zone
causing significant surface deformation. Detailed event information can be found
at the [USGS](https://earthquake.usgs.gov/earthquakes/eventpage/us60007ewc/executive)
report.

Process an Sentinel-1 interferogram to generate the co-seismic deformation field
of this earthquake and its corresponding quality and geometry information.


# Appendix: OPERA CSLC-S1 Product Reference



## A.1 Product Overview

**OPERA_L2_CSLC-S1** (Coregistered Single-Look Complex from Sentinel-1)
is a Level-2 SAR product produced by the OPERA project at JPL/NASA
(*OPERA Project, [JPL](https://www.jpl.nasa.gov/go/opera/products/cslc-product-suite/), 2023*).
Each CSLC image contains amplitude and phase of the complex radar return,
coregistered and geocoded onto a common UTM grid.

| Property | Value |
|----------|-------|
| Product level | L2 (Level-2) |
| Format | HDF5 |
| Pixel type | complex64 |
| East posting | 5 m |
| North posting | 10 m |
| DAAC | ASF DAAC |
| Coverage | North America (US, US Territories, Canada within 200 km of US border, through Panama) |

**Applications:** InSAR time-series analysis, input for OPERA DISP-S1
displacement product, persistent/distributed scatterer processing,
coherence-based change detection.


## A.2 File Naming Convention

CSLC-S1 products follow the OPERA naming convention defined in the
*OPERA CSLC-S1 Product Specification* (D-108278, \u00a75.1):

```
OPERA_L2_CSLC-S1_<BURST-ID>_<ACQ-TIME>_<GEN-TIME>_<VERSION>.h5
```

| Segment | Format | Example |
|---------|--------|---------|
| `PROJECT` | `OPERA` | OPERA |
| `LEVEL` | `L2` | L2 |
| `PRODUCT-TYPE` | `CSLC-S1` | CSLC-S1 |
| `BURST-ID` | `tRRR_BBBBBB_iwN` | `t124_264305_iw2` |
| `ACQ-TIME` | `YYYYMMDDTHHMMSSZ` | `20240915T043110Z` |
| `GEN-TIME` | `YYYYMMDDTHHMMSSZ` | `20241001T120000Z` |
| `VERSION` | `vX.Y` | `v1.1` |

**Companion Product &mdash; CSLC-S1-STATIC**:
`OPERA_L2_CSLC-S1-STATIC` contains static radar geometry layers
(layover/shadow mask, local incidence angle, LOS vectors) generated
once per unique burst ID (*OPERA CSLC-S1 ATBD*, D-108752, \u00a73).
These layers are used in &sect; 6 to compute LOS incidence and azimuth
angles for the stitched interferogram.


## A.3 HDF5 Dataset Layout

The HDF5 file is organised into four groups (*CSLC-S1 Product
Specification*, D-108278, \u00a74):

**`/data/`** &mdash; primary data arrays:

| Dataset | Type | Dim | Description |
|---------|------|-----|-------------|
| `VV` | complex64 | (rows, cols) | Geocoded single-look complex SAR |
| `x_coordinates` | float64 | (cols,) | UTM easting (m) |
| `y_coordinates` | float64 | (rows,) | UTM northing (m) |
| `projection` | int32 | scalar | EPSG code |

**`/identification/`** &mdash; file-level metadata: burst ID, mission
identifier, orbit/timing, bounding polygon.

**`/metadata/processing_information/timing_corrections/`** &mdash;
correction LUT grids used in &sect; 4.3 (120 m ground spacing):

| Dataset | Description |
|---------|-------------|
| `slant_range` | Slant-range LUT (m) |
| `zero_doppler_time` | Zero-Doppler time LUT (s) |
| `bistatic_delay` | Bistatic delay (s) |
| `los_solid_earth_tides` | Solid Earth tide (m LOS) |
| `los_ionospheric_delay` | Ionospheric delay (m LOS) |
| `wet_los_troposphere_delay` | Wet troposphere (m LOS) |
| `dry_los_troposphere_delay` | Dry troposphere (m LOS) |

**`/quality_assurance/`** &mdash; per-dataset statistics: min, max,
mean, standard deviation.


# Relevant references:

- De Zan, F., & Guarnieri, A. M. (2006). TOPSAR: Terrain observation by progressive scans. IEEE Trans. Geosci. Remote Sens., 44(9), 2352-2360. https://doi.org/10.1109/TGRS.2006.873853

- Fattahi, H., Agram, P., & Simons, M. (2016). A Network-Based Enhanced Spectral Diversity Approach for TOPS Time-Series Analysis. IEEE Trans. Geosci. Remote Sens., 55(2), 777-786. https://doi.org/10.1109/TGRS.2016.2614925

- Fattahi, H., Brancato, V., Jeong, S., Yunjun, Z., Staniewicz, S., Bekaert, D., et al. (2022). OPERA Coregistered Single Look Complex products from Sentinel-1 data. AGU Fall Meeting 2022, Chicago, IL, USA. https://doi.org/10.48577/jpl.4ZPEJL

- Gisinger, C., Schubert, A., Breit, H., Garthwaite, M., Balss, U., Willberg, M., et al. (2021). In-Depth Verification of Sentinel-1 and TerraSAR-X Geolocation Accuracy Using the Australian Corner Reflector Array. IEEE Trans. Geosci. Remote Sens., 59(2), 1154-1181. https://doi.org/10.1109/TGRS.2019.2961248

- Gisinger, C., Libert, L., Marinkovic, P., Krieger, L., Larsen, Y., Valentino, A., et al. (2022). The Extended Timing Annotation Dataset for Sentinel-1 — Product Description and First Evaluation Results. IEEE Trans. Geosci. Remote Sens., 60. https://doi.org/10.1109/TGRS.2022.3194216

- Liang, C., Agram, P., Simons, M., & Fielding, E. J. (2019). Ionospheric Correction of InSAR Time Series Analysis of C-band Sentinel-1 TOPS Data. IEEE Trans. Geosci. Remote Sens., 59(9), 6755 - 6773. https://doi.org/10.1109/TGRS.2019.2908494

- OPERA CSLC-S1 Algorithm Theoretical Basis Document (ATBD), D-108752, NASA JPL, 2024. <https://cumulus.asf.earthdatacloud.nasa.gov/PUBLIC/DATA/OPERA/OPERA_CSLC-S1_ATBD_D-108752_Initial_2024-06-24_signed.pdf>

- OPERA CSLC-S1 Product Specification, D-108278, v1.0, NASA JPL, 2023. <https://d2pn8kiwq2w21t.cloudfront.net/documents/OPERA_CSLC-S1_ProductSpec_v1.0.0_D-108278_Initial_2023-09-11_URS321269.pdf>

- OPERA CSLC-S1 Product Suite &mdash; JPL Operational Product Page. <https://www.jpl.nasa.gov/go/opera/products/cslc-product-suite/>

- Sansosti, E., Berardino, P., Manunta, M., Serafino, F., & Fornaro, G. (2006). Geometrical SAR image registration. IEEE Trans. Geosci. Remote Sens., 44(10), 2861-2870. https://doi.org/10.1109/TGRS.2006.875787

- Yunjun, Z., Fattahi, H., Pi, X., Rosen, P., Simons, M., Agram, P., & Aoki, Y. (2022). Range Geolocation Accuracy of C-/L-Band SAR and its Implications for Operational Stack Coregistration. IEEE Trans. Geosci. Remote Sens., 60, 5227219. https://doi.org/10.1109/TGRS.2022.3168509

- Zebker, H. A. (2017). User-Friendly InSAR Data Products: Fast and Simple Timeseries Processing. IEEE Geosci. Remote Sens. Lett., 14(11), 2122-2126. https://doi.org/10.1109/LGRS.2017.2753580

- Zheng, Y., & Zebker, H. A. (2017). Phase Correction of Single-Look Complex Radar Images for User-Friendly Efficient Interferogram Formation. IEEE J. Sel. Topics Appl. Earth Observ., 10(6), 2694-2701. https://doi.org/10.1109/JSTARS.2017.2697861

